In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
    'dealerstampcreation__app',
]

Project: 20231010-gen-xii


### Output

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Load ```df_cols_bad.csv```

In [4]:
%%time

# import
str_filename = 'df_cols_bad.csv'
str_local_path = f'../03_leaky/output/{str_filename}'
df = pd.read_csv(str_local_path)
list_leaky = list(df['feature'])
list_leaky = [col for col in list_leaky if col not in list_cols_id]

# message
print(f'There are {len(list_leaky)} bad columns')
print('')

There are 925 bad columns

CPU times: user 2.59 ms, sys: 263 µs, total: 2.85 ms
Wall time: 2.66 ms


### Get list of original columns

In [5]:
%%time

# get cols
str_filename = 'df_descriptives_train.csv'
str_local_path = f'../../04_eda_pt_2/output/{str_filename}'
list_cols = list(pd.read_csv(str_local_path)['feature'])

# rm leaky
list_cols = [col for col in list_cols if col not in list_leaky]

# ensure ids
list_cols = list_cols + list_cols_id
# rm dups
list_cols = list(dict.fromkeys(list_cols))

# message
print(f'There are {len(list_cols)} non-leaky features (including necessary ID columns)')
print('')

There are 1567 non-leaky features (including necessary ID columns)

CPU times: user 24.8 ms, sys: 7.69 ms, total: 32.5 ms
Wall time: 32 ms


### Save df

In [6]:
%%time

# make df
df = pd.DataFrame({'feature': list_cols})

# save
str_filename = 'df_non_leaky.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

# show
df

CPU times: user 2.56 ms, sys: 0 ns, total: 2.56 ms
Wall time: 2.26 ms


,feature
0,linkf060__tu
1,linkf045__tu
2,linkf185__tu
3,linkf193__tu
4,linkf105__tu
...,...
1562,intservicecontractmileageaddon__app
1563,bitmaintenanceagreement__app
1564,uniqueid
1565,applicationdate__app


### Create no leaks dfs

In [7]:
%%time

# import, subset, and write
for str_df in tqdm (['train','valid','test']):
    # set names
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
    # read from s3
    df = pd.read_parquet(str_uri, columns=list_cols)
    # replace
    df.replace(['NaN','nan'], np.nan, inplace=True)
    
    # create year_month col
    df['year_month'] = df[str_datecol].astype(str).str[:7]
    
    # mark data set
    df['data_set'] = str_df

    # write to s3
    str_filename = f'df_{str_df}_noleaks.gzip'
    str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
    df.to_parquet(str_uri, compression='gzip')

100%|██████████| 3/3 [00:45<00:00, 15.32s/it]

CPU times: user 51.6 s, sys: 22 s, total: 1min 13s
Wall time: 46 s
